In [3]:
import json
import pandas as pd

# Load the entire fixed file as a single JSON array
with open('dataset.jsonl') as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(df.head())


         id                                        description  \
0  sqli-001                Basic tautology-based SQL injection   
1  sqli-002          Union-based SQL injection to extract data   
2  sqli-003                Blind SQL injection with time delay   
3  sqli-004  Error-based SQL injection to reveal database v...   
4  sqli-005                  Boolean-based blind SQL injection   

                                          payload                    context  \
0                                     ' OR '1'='1  Login form username input   
1  ' UNION SELECT username, password FROM users--         Search input field   
2                      '; WAITFOR DELAY '0:0:5'--        ID parameter in URL   
3                ' AND 1=CONVERT(int,@@version)--              User ID input   
4                                     ' AND 1=1--               Search field   

            type severity                                      example_query  \
0      tautology     high  SELECT * FROM u

In [5]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['severity_encoded'] = le.fit_transform(df['severity'])

print(df[['severity', 'severity_encoded']].head())

  severity  severity_encoded
0     high                 1
1     high                 1
2   medium                 3
3   medium                 3
4   medium                 3


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Remove rows where payload is missing
df = df.dropna(subset=['payload'])

# Convert payload column to string
df['payload'] = df['payload'].astype(str)

# TF-IDF
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X = vectorizer.fit_transform(df['payload'])

y = df['severity_encoded']

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [9]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

models = {

    "RandomForest": {
        "model": RandomForestClassifier(),
        "params": {
            "n_estimators": [100, 200],
            "max_depth": [10, 20, None]
        }
    },

    "SVM": {
        "model": SVC(),
        "params": {
            "C": [1, 10],
            "kernel": ["linear", "rbf"]
        }
    },

    "LogisticRegression": {
        "model": LogisticRegression(max_iter=2000),
        "params": {
            "C": [0.1, 1, 10]
        }
    },

    "DecisionTree": {
        "model": DecisionTreeClassifier(),
        "params": {
            "max_depth": [5, 10, None]
        }
    },

    "XGBoost": {
        "model": XGBClassifier(eval_metric='mlogloss'),
        "params": {
            "n_estimators": [100, 200],
            "max_depth": [3, 5, 7],
            "learning_rate": [0.01, 0.1]
        }
    }
}

In [10]:
results = []

best_model = None
best_accuracy = 0
best_model_name = ""

for model_name, mp in models.items():

    print(f"\nTraining {model_name}...")

    grid = GridSearchCV(
        estimator=mp['model'],
        param_grid=mp['params'],
        cv=5,
        scoring='accuracy',
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    trained_model = grid.best_estimator_

    y_pred = trained_model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)

    print("Accuracy:", acc)
    print("Best Params:", grid.best_params_)

    results.append({
        'Model': model_name,
        'Accuracy': acc,
        'Best Params': grid.best_params_
    })

    if acc > best_accuracy:
        best_accuracy = acc
        best_model = trained_model
        best_model_name = model_name


Training RandomForest...
Accuracy: 0.81
Best Params: {'max_depth': None, 'n_estimators': 200}

Training SVM...
Accuracy: 0.81
Best Params: {'C': 10, 'kernel': 'linear'}

Training LogisticRegression...
Accuracy: 0.8
Best Params: {'C': 10}

Training DecisionTree...
Accuracy: 0.84
Best Params: {'max_depth': None}

Training XGBoost...
Accuracy: 0.83
Best Params: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200}


In [11]:
import pandas as pd

results_df = pd.DataFrame(results)

print(results_df)

print("\nBest Model:", best_model_name)
print("Best Accuracy:", best_accuracy)

                Model  Accuracy  \
0        RandomForest      0.81   
1                 SVM      0.81   
2  LogisticRegression      0.80   
3        DecisionTree      0.84   
4             XGBoost      0.83   

                                         Best Params  
0           {'max_depth': None, 'n_estimators': 200}  
1                      {'C': 10, 'kernel': 'linear'}  
2                                          {'C': 10}  
3                                {'max_depth': None}  
4  {'learning_rate': 0.1, 'max_depth': 7, 'n_esti...  

Best Model: DecisionTree
Best Accuracy: 0.84


In [12]:
from sklearn.metrics import confusion_matrix, classification_report

final_pred = best_model.predict(X_test)

print(confusion_matrix(y_test, final_pred))

print(classification_report(y_test, final_pred))

[[12  2  0  2]
 [ 2 43  0  4]
 [ 0  0  1  2]
 [ 1  3  0 28]]
              precision    recall  f1-score   support

           0       0.80      0.75      0.77        16
           1       0.90      0.88      0.89        49
           2       1.00      0.33      0.50         3
           3       0.78      0.88      0.82        32

    accuracy                           0.84       100
   macro avg       0.87      0.71      0.75       100
weighted avg       0.85      0.84      0.84       100



In [13]:
import pickle

with open("payload_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

In [14]:
with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

In [15]:
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

In [16]:
new_payloads = [
    "' OR '1'='1",
    "<script>alert('xss')</script>",
    "normal user input"
]

# Convert text to TF-IDF features
X_new = vectorizer.transform(new_payloads)

# Predict
predictions = best_model.predict(X_new)

# Convert numbers back to labels
labels = le.inverse_transform(predictions)

for payload, label in zip(new_payloads, labels):
    print("Payload:", payload)
    print("Predicted Severity:", label)
    print()

Payload: ' OR '1'='1
Predicted Severity: high

Payload: <script>alert('xss')</script>
Predicted Severity: high

Payload: normal user input
Predicted Severity: medium

